# 40_00 - Phase 4 Overview, Portability Checks, and Pretrained CNN Readiness

This notebook does **not** train a model.

It verifies that the Phase 4 pretrained-CNN pipeline is ready to run safely across devices and operating systems.

It is designed to be:

- Run All safe
- portable across Windows and Linux
- aligned with the shared data pipeline from earlier phases
- lightweight enough to rerun before starting any model notebook
        

In [ ]:
from pathlib import Path
import json
import os
import sys

import mlflow
import pandas as pd

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()


def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "requirements.txt"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers_all) and any((candidate / marker).exists() for marker in markers_any):
            return candidate
    raise RuntimeError(f"Could not locate project root from: {start}")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DATA_DIR = PROJECT_ROOT / "data"
CONFIGS_DIR = PROJECT_ROOT / "configs"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
TRACKING_DIR = PROJECT_ROOT / "mlruns"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
SPLIT_DIR = DATA_DIR / "splits" / "split_v1"
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"
CLASSES_JSON = SPLIT_DIR / "classes.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("REPORTS_METRICS_DIR:", REPORTS_METRICS_DIR)
print("TRACKING_DIR       :", TRACKING_DIR)

In [ ]:
from src.data.dataset_loader import ImageDataset
from src.data.transforms import load_transforms_config, get_train_transforms, get_eval_transforms
from src.models.cnn_pretrained import build_model, count_parameters, get_weights_name, list_available_models

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_config": TRANSFORMS_CONFIG_PATH,
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs for Phase 4 overview: {missing}")

REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    classes_to_idx = json.load(f)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

cfg = load_transforms_config(TRANSFORMS_CONFIG_PATH)
train_tfm = get_train_transforms(cfg)
eval_tfm = get_eval_transforms(cfg)

train_ds = ImageDataset(
    split_csv=TRAIN_CSV,
    transform=train_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
val_ds = ImageDataset(
    split_csv=VAL_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)

model_rows = []
for model_name in list_available_models().keys():
    model = build_model(model_name=model_name, num_classes=len(classes_to_idx), pretrained=False)
    model_rows.append(
        {
            "model_name": model_name,
            "weights_name": get_weights_name(model_name=model_name, pretrained=True),
            "parameter_count": int(count_parameters(model)),
        }
    )

summary = {
    "phase": "phase_4_overview",
    "status": "ready",
    "project_root": str(PROJECT_ROOT),
    "split_id": "split_v1",
    "transforms_config": str(TRANSFORMS_CONFIG_PATH),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "dataset_lengths": {"train": len(train_ds), "val": len(val_ds)},
    "device_default": "cuda" if __import__("torch").cuda.is_available() else "cpu",
    "supported_models": model_rows,
}
summary_path = REPORTS_METRICS_DIR / "phase4_overview_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("[OK] Saved overview summary to:", summary_path)

In [ ]:
with mlflow.start_run(run_name="phase4_overview_validation"):
    mlflow.log_param("stage", "phase4_overview")
    mlflow.log_param("split_id", "split_v1")
    mlflow.log_param("num_supported_models", len(summary["supported_models"]))
    mlflow.log_metric("train_size", summary["train_size"])
    mlflow.log_metric("val_size", summary["val_size"])
    mlflow.log_metric("test_size", summary["test_size"])
    mlflow.log_artifact(str(summary_path))

print("[OK] MLflow logging complete.")